# Step1: Data Quality Report
## Load Dataset

In [1]:
import pandas as pd
import numpy as np
import re
df=pd.read_csv("global_freelancers_raw.csv")
# print(df)
df.head(10)
df.info()
df=df.drop(columns=["is_active", "client_satisfaction"])
df

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   freelancer_ID        1000 non-null   str    
 1   name                 1000 non-null   str    
 2   gender               1000 non-null   str    
 3   age                  970 non-null    float64
 4   country              1000 non-null   str    
 5   language             1000 non-null   str    
 6   primary_skill        1000 non-null   str    
 7   years_of_experience  949 non-null    float64
 8   hourly_rate (USD)    906 non-null    str    
 9   rating               899 non-null    float64
 10  is_active            911 non-null    str    
 11  client_satisfaction  824 non-null    str    
dtypes: float64(3), str(9)
memory usage: 93.9 KB


,freelancer_ID,name,gender,age,country,language,primary_skill,years_of_experience,hourly_rate (USD),rating
0,FL250001,Ms. Nicole Kidd,f,52.0,Italy,Italian,Blockchain Development,11.0,100,NaN
1,FL250002,Vanessa Garcia,FEMALE,52.0,Australia,English,Mobile Apps,34.0,USD 100,3.3
2,FL250003,Juan Nelson,male,53.0,Germany,German,Graphic Design,31.0,50,0.0
3,FL250004,Amanda Spencer,F,38.0,Australia,English,Web Development,4.0,$40,1.5
4,FL250005,Lynn Curtis DDS,female,53.0,Germany,German,Web Development,27.0,30,4.8
...,...,...,...,...,...,...,...,...,...,...
995,FL250996,Albert Wilcox,Male,56.0,Turkey,Turkish,DevOps,13.0,100,0.0
996,FL250997,Cheryl Norris,f,26.0,Germany,German,Blockchain Development,6.0,USD 40,2.8
997,FL250998,Kathy Watkins,female,37.0,Japan,Japanese,Data Analysis,15.0,75,NaN
998,FL250999,John Obrien,m,46.0,Russia,Russian,Machine Learning,22.0,100,2.8


## Count Null Values 

In [2]:

null_count=pd.DataFrame({
    'Total_null' : df.isna().sum(),
    'Null_prctg' : (df.isna().mean()*100).round(2)
}).sort_values('Total_null', ascending=False)
null_count

,Total_null,Null_prctg
rating,101,10.1
hourly_rate (USD),94,9.4
years_of_experience,51,5.1
age,30,3.0
name,0,0.0
freelancer_ID,0,0.0
language,0,0.0
country,0,0.0
gender,0,0.0
primary_skill,0,0.0


**Understand Output:** 
- This Null Count per columns output shows that `client_satisfaction` has total 117 null entries and `rating` has 101 followed by `hourly_rate(USD)`, `is_active`, `years_of_experience` and `age` are the columns having missing data.
- While (`freelancer_ID`, `name`, `gender`, `country`, `language`, `primary_skill`) are fully populated.

### Check Duplicates Rows

In [3]:
duplicate_rows= df.duplicated().sum
id_duplicates=df["freelancer_ID"].sum()

print(f"Exact Duplicates rows:", duplicate_rows)
print(f"Exact Freelancer_ID duplicate Values: {id_duplicates}" )

Exact Duplicates rows: <bound method Series.sum of 0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Length: 1000, dtype: bool>
Exact Freelancer_ID duplicate Values: FL250001FL250002FL250003FL250004FL250005FL250006FL250007FL250008FL250009FL250010FL250011FL250012FL250013FL250014FL250015FL250016FL250017FL250018FL250019FL250020FL250021FL250022FL250023FL250024FL250025FL250026FL250027FL250028FL250029FL250030FL250031FL250032FL250033FL250034FL250035FL250036FL250037FL250038FL250039FL250040FL250041FL250042FL250043FL250044FL250045FL250046FL250047FL250048FL250049FL250050FL250051FL250052FL250053FL250054FL250055FL250056FL250057FL250058FL250059FL250060FL250061FL250062FL250063FL250064FL250065FL250066FL250067FL250068FL250069FL250070FL250071FL250072FL250073FL250074FL250075FL250076FL250077FL250078FL250079FL250080FL250081FL250082FL250083FL250084FL250085FL250086FL250087FL250088FL250089FL250090FL250091FL250092FL250

***Understanding this Output** 
- After reading this output, its shows there is not even a single duplicate row in in 1000 rows.

### Data types of Columns:

In [4]:
print(df.dtypes)

freelancer_ID              str
name                       str
gender                     str
age                    float64
country                    str
language                   str
primary_skill              str
years_of_experience    float64
hourly_rate (USD)          str
rating                 float64
dtype: object


### Value Range Anomolies:

- First check Numeric range anomoly, whether the number falls within the realistic range for what's measuring.

In [5]:
numeric_columns= "age, year_of_experience, hourly_rate(USD), rating"

print("---- age ----")
print(df['age'].describe())
print()

print("----years_of_experience----")
print(df["years_of_experience"].describe())
print()

print("----rating----")
print(df["rating"].describe())
print()

# print("----hourly_rates----")
# print(df["hourly_rate (USD)"].describe())

---- age ----
count    970.000000
mean      40.509278
std       11.942605
min       20.000000
25%       31.000000
50%       41.000000
75%       51.000000
max       60.000000
Name: age, dtype: float64

----years_of_experience----
count    949.000000
mean      11.340358
std        9.680610
min        0.000000
25%        3.000000
50%        9.000000
75%       17.000000
max       41.000000
Name: years_of_experience, dtype: float64

----rating----
count    899.000000
mean       2.512570
std        1.546599
min        0.000000
25%        1.400000
50%        2.600000
75%        3.800000
max        5.000000
Name: rating, dtype: float64



In [6]:
## Categorical Anomly check
print("Gender - Raw Values Count")
print(df["gender"].value_counts(dropna=False))
print()
print(f"Number of Unique spellings found: {df["gender"].unique()}" )
print("Expected 2: (males/females)")

Gender - Raw Values Count
gender
FEMALE    115
M         106
f         103
Male      103
MALE      102
male      100
m          99
Female     96
F          90
female     86
Name: count, dtype: int64

Number of Unique spellings found: <StringArray>
['f', 'FEMALE', 'male', 'F', 'female', 'm', 'MALE', 'Female', 'M', 'Male']
Length: 10, dtype: str
Expected 2: (males/females)


***Understanding Anomolies Output***
- `age` gives maximum value `60` and minimum `20` which is plausible not beyond real world.
- `years_of_experience` also shows minimum `0` and max `41` which also plausible and bound within real-world and not negative.
- `rating` is bounded 0–5 as expected, with a real (non-null) minimum of exactly 0
- `gender` is the clearest anomly in dataset as ***10 different spellings*** for same thing (`f`, `F`, `Female`, `female`, `FEMALE`, `m`, `Male`, `male`, `MALE`, `M`) representing only ***2 Real Categories***.

### Data Quality Report Summary 

| Check | Finding |
|---|---|
| Null values | 4 of 10 columns affected: `rating` (10.1%), `hourly_rate (USD)` (9.4%), `years_of_experience` (5.1%), `age` (3.0%) |
| Duplicate rows | 0 exact duplicates, 0 duplicate IDs |
| Data type issues | 1 genuine fix needed: `hourly_rate (USD)` is text with 3 mixed formats and must be converted to float |
| Value range anomalies (numeric) | None — `age`, `years_of_experience`, `rating` all fall within plausible bounds; 0 logically-impossible age/experience combinations |
| Value range anomalies (categorical) | `gender` has 10 inconsistent spellings for 2 real categories need to standarize |

This dataset's problems are concentrated in **missing data** and **inconsistent formatting**, not in duplicates or implausible values. That shapes the priorities for the rest of the project: Steps 2 (missing data) and 4 (standardization) will carry most of the actual cleaning work; Step 5 (outlier detection) is expected to find little to nothing, and that will be reported honestly rather than padded out.

## Step:2   Missing Data Handling

In [7]:
print("age skewness: ", df['age'].skew().round(3))
df['age'].describe()

age skewness:  -0.097


count    970.000000
mean      40.509278
std       11.942605
min       20.000000
25%       31.000000
50%       41.000000
75%       51.000000
max       60.000000
Name: age, dtype: float64

***Decision: Median Imputation*** 
- skewness is -0.10 as both *Mean* and *Median* would land nearly on the same place because they are symmetric.
- *Median* is chosen BTW as the more defensible default for an identifier-adjacent human attribute like age: mean of 40.5 doesn't exists but a mode of 41  does. 

In [8]:
median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age)
df["age"] = df["age"].astype(int)

print(f"Imputed value of age: ", median_age)
print(f"Remaining nulls in age: ", df["age"].isna().sum())

Imputed value of age:  41.0
Remaining nulls in age:  0


## Year's Of Experience (Median Imputation)

In [9]:
print("years_of_experience skewness: ", df["years_of_experience"].skew().round(3))
df["years_of_experience"].describe()

years_of_experience skewness:  0.89


count    949.000000
mean      11.340358
std        9.680610
min        0.000000
25%        3.000000
50%        9.000000
75%       17.000000
max       41.000000
Name: years_of_experience, dtype: float64

***Decision: Median Imputation*** 
- skewness is positive **0.89** mostly freelancers have few years of experience.
- with this skewness, mean would flated above where most of the data actually sits. *Median* is the best choice and it fits here.

In [10]:
medain_experience = df["years_of_experience"].median()
df["years_of_experience"]=df["years_of_experience"].fillna(medain_experience)
df["years_of_experience"]=df["years_of_experience"].astype(int)

print(f"Imputed value of Years of Experience: ", medain_experience)
print(f"Remaining nulls in years_of_experience: ", df["years_of_experience"].isna().sum())

Imputed value of Years of Experience:  9.0
Remaining nulls in years_of_experience:  0


### Hourly Rate(USD) (9.4% missing) → group-median imputation ###

In [11]:

def parse_rate(value):
    if pd.isna(value):
        return np.nan
    cleaned = re.sub(r'[^0-9.]', '', str(value))
    return float(cleaned) if cleaned else np.nan

df['hourly_rate (USD)'] = df['hourly_rate (USD)'].apply(parse_rate)

print("hourly_rate skewness (non-null):", df['hourly_rate (USD)'].skew().round(3))
print()
print("Median rate by primary_skill:")
print(df.groupby('primary_skill')['hourly_rate (USD)'].median().sort_values())

hourly_rate skewness (non-null): 0.625

Median rate by primary_skill:
primary_skill
AI                        40.0
Blockchain Development    40.0
Data Analysis             40.0
Graphic Design            40.0
Mobile Apps               40.0
Machine Learning          40.0
Web Development           40.0
Cybersecurity             50.0
DevOps                    50.0
UI/UX Design              50.0
Name: hourly_rate (USD), dtype: float64


In [12]:
df["hourly_rate (USD)"]=df.groupby("primary_skill")['hourly_rate (USD)'] \
    .transform(lambda x: x.fillna(x.median()))
print(f"Remaining nulls in hourly_rate (USD):", df["hourly_rate (USD)"].isna().sum())
df["hourly_rate (USD)"].describe()

Remaining nulls in hourly_rate (USD): 0


count    1000.000000
mean       51.580000
std        26.188536
min        20.000000
25%        30.000000
50%        40.000000
75%        75.000000
max       100.000000
Name: hourly_rate (USD), dtype: float64

### Rating (10.1% missing) -> **imputed Median** ###
This is the one column where imputation would actively introduce a wrong answer, not just an imprecise one.

In [13]:
print("NaN count: ", df['rating'].isna().sum())
print("Exact 0.0 count: ", (df['rating']==0.0).sum())
print()
print(df['rating'].describe())

NaN count:  101
Exact 0.0 count:  145

count    899.000000
mean       2.512570
std        1.546599
min        0.000000
25%        1.400000
50%        2.600000
75%        3.800000
max        5.000000
Name: rating, dtype: float64


**Decision:** leave `NaN` as `NaN`. Do not mean/median impute. Add a `has_rating` flag column instead.

In [14]:
df["has_rating"]=df["rating"].notna()
print(df['has_rating'].value_counts())
print()
print(f"rating columns left untouched-still {df['rating'].isna().sum()}, NaNs by design")

has_rating
True     899
False    101
Name: count, dtype: int64

rating columns left untouched-still 101, NaNs by design


### Missing Data Handling Summary

| Column | % missing | Strategy | Why |
|---|---|---|---|
| `age` | 3.0% | Median imputation | Near-symmetric distribution; median avoids a non-occurring fractional value |
| `years_of_experience` | 5.1% | Median imputation | Right-skewed (0.89); mean would overstate typical experience |
| `hourly_rate (USD)` | 9.4% | Median imputation, grouped by `primary_skill` | Right-skewed, and rate genuinely varies by skill; a global median would distort skill-specific pay levels |
| `rating` | 10.1% | **No imputation** left as `NaN`, flagged with new `has_rating` column | `NaN` and `0.0` are meaningfully different (unrated vs. rated-zero); imputing would fabricate a reputation for unrated freelancers |

Row deletion wasn't used anywhere; missingness never exceeded ~10% in any column, and there was no case where imputation would have been more misleading than removing the data outright.

## Step:3 Duplicate Removal

Step 1 already checked for duplicates on the raw data and found none. So there is no need of code for duplicate rows removal.

In [15]:
df=df.drop_duplicates()
df

,freelancer_ID,name,gender,age,country,language,primary_skill,years_of_experience,hourly_rate (USD),rating,has_rating
0,FL250001,Ms. Nicole Kidd,f,52,Italy,Italian,Blockchain Development,11,100.0,NaN,False
1,FL250002,Vanessa Garcia,FEMALE,52,Australia,English,Mobile Apps,34,100.0,3.3,True
2,FL250003,Juan Nelson,male,53,Germany,German,Graphic Design,31,50.0,0.0,True
3,FL250004,Amanda Spencer,F,38,Australia,English,Web Development,4,40.0,1.5,True
4,FL250005,Lynn Curtis DDS,female,53,Germany,German,Web Development,27,30.0,4.8,True
...,...,...,...,...,...,...,...,...,...,...,...
995,FL250996,Albert Wilcox,Male,56,Turkey,Turkish,DevOps,13,100.0,0.0,True
996,FL250997,Cheryl Norris,f,26,Germany,German,Blockchain Development,6,40.0,2.8,True
997,FL250998,Kathy Watkins,female,37,Japan,Japanese,Data Analysis,15,75.0,NaN,False
998,FL250999,John Obrien,m,46,Russia,Russian,Machine Learning,22,100.0,2.8,True


## Step:4  Standardization

Checking every text column for inconsistent formatting: mixed casing, extra whitespace, or the same real-world value spelled multiple ways.


In [16]:
print("Column Data Types, confirming no date type or date like column exists.")
print(df.dtypes)

Column Data Types, confirming no date type or date like column exists.
freelancer_ID              str
name                       str
gender                     str
age                      int64
country                    str
language                   str
primary_skill              str
years_of_experience      int64
hourly_rate (USD)      float64
rating                 float64
has_rating                bool
dtype: object


### Gender: 9 Consistent spelling Mistakes
- This is the column which was flagged back the in value range check. 

In [17]:
print("Before Standardization.")
print(df['gender'].value_counts(dropna=False))


Before Standardization.
gender
FEMALE    115
M         106
f         103
Male      103
MALE      102
male      100
m          99
Female     96
F          90
female     86
Name: count, dtype: int64


In [18]:
gender_map={
    'Male': 'Male', 'MALE': 'Male', 'male': 'Male', 'm': 'Male', 'M': 'Male',
    'Female': 'Female', 'female': 'Female', 'f': 'Female', 'FEMALE': 'Female', 'F': 'Female'
}
df['gender']=df['gender'].map(gender_map)

print("After Standardization")
print(df['gender'].value_counts(dropna=False))

After Standardization
gender
Male      510
Female    490
Name: count, dtype: int64


***Observations*** 
- after standardizing `gender` column only 2 names `Male` and `Female` are left.

### Standardizing other text columns `country`,`language`,`primary_skill`.
checked for the same problem (mixed spacing, whitespace, spelling variant) that `gender` had.

In [19]:
for col in ["country", "language", "primary_skill"]:
    has_whitespace_issue=(df[col].astype(str) != (df[col].astype(str).str.strip().sum()))
    print(f"{col}: {df[col].unique()} unique values, {has_whitespace_issue} rows with leading whitespace" )
    print(sorted(df[col].unique()))
    print()

country: <StringArray>
[         'Italy',      'Australia',        'Germany',    'Netherlands',
      'Indonesia',  'United States',         'Turkey', 'United Kingdom',
      'Argentina',          'Japan',          'India',         'Brazil',
    'South Korea',         'Russia',         'Canada',         'France',
          'Egypt',   'South Africa',          'China',         'Mexico',
          'Spain']
Length: 21, dtype: str unique values, 0      True
1      True
2      True
3      True
4      True
       ... 
995    True
996    True
997    True
998    True
999    True
Name: country, Length: 1000, dtype: bool rows with leading whitespace
['Argentina', 'Australia', 'Brazil', 'Canada', 'China', 'Egypt', 'France', 'Germany', 'India', 'Indonesia', 'Italy', 'Japan', 'Mexico', 'Netherlands', 'Russia', 'South Africa', 'South Korea', 'Spain', 'Turkey', 'United Kingdom', 'United States']

language: <StringArray>
[   'Italian',    'English',     'German',      'Dutch', 'Indonesian',
    'Turkis

***Findings:*** No Standandization needs for these 3 columns.
- `country` (21 values), `language` (16 values), `primary_skill` (10 values) are all already cased and spelled with zero whitespace issues. 

### `hourly_rate (USD)` needs to be standardize:
- It is already standardize in the above **step 2 Missing Data Handling**. no need to standardize here.

In [20]:
print("Hourly rate (USD) dtype:", df["hourly_rate (USD)"].dtype)
print("Sample values:", df['hourly_rate (USD)'].head(4).tolist())

Hourly rate (USD) dtype: float64
Sample values: [100.0, 100.0, 50.0, 40.0]


###  Standardization Summary

| Column | Issue found | Action |
|---|---|---|
| `gender` | 9 spellings for 2 categories | Mapped to `Male` / `Female` |
| `country` | None | already clean |
| `language` | None | already clean |
| `primary_skill` | None | already clean |
| `hourly_rate (USD)` | Mixed text formats | Already resolved in Step 2 (parsed to numeric ahead of imputation) |
| Date columns | N/A | No date column exists in this dataset |
| `name` | Embedded titles/suffixes in a few rows | Flagged, not fixed, depends on downstream use, left as an open decision |

`gender` was the only column in this dataset that actually needed standardization work. The rest of the checklist item is reported as "checked, clean" rather than padded with unnecessary fixes.

## Step:5 Outlier Detection Using IQR Technique
Using IQR technique as primary because step 2 already established that `years_of_experience (0.89)` and `hourly_rate (USD) (0.62)` are right-skewed. And Z-score relies on mean and Standard Daviation which are themselves distorted by skewed. While IQR relies on Quartiles which are not pulled around extreme values like in Mean.

- **Check On** `age`, `years_of_experience`, `hourly_rate (USD)` and `rating` as they are numeric columns.

In [21]:
def IQR_outlier(series):
    q1=series.quantile(0.25)
    q3=series.quantile(0.75)
    IQR=q3-q1
    lower_bound= q1 - (1.5 * IQR)
    upper_bound= q3 + (1.5 * IQR)
    outliers=series[(series<lower_bound)|(series>upper_bound)]
    return lower_bound, upper_bound, outliers
numeric_cols=('age', 'years_of_experience', 'hourly_rate (USD)', 'rating')

for col in numeric_cols:
    lower, upper, outliers=IQR_outlier(df[col].dropna())
    print(f"------{col}------")
    print(f"Valid range (IQR bounds), {lower:.1f} to {upper:.1f}")
    print(f"outliers found: {len(outliers)}")
    if len(outliers)>0:
        print(f"Outliers values: {sorted(outliers.unique())}")
    print()

------age------
Valid range (IQR bounds), 1.0 to 81.0
outliers found: 0

------years_of_experience------
Valid range (IQR bounds), -18.0 to 38.0
outliers found: 9
Outliers values: [np.int64(39), np.int64(40), np.int64(41)]

------hourly_rate (USD)------
Valid range (IQR bounds), -37.5 to 142.5
outliers found: 0

------rating------
Valid range (IQR bounds), -2.2 to 7.4
outliers found: 0



**Results:**

| Column | IQR bounds | Outliers found |
|---|---|---|
| `age` | 1.0 – 81.0 | 0 |
| `years_of_experience` | -18.0 – 38.0 | **9** (values of 39, 40, 41) |
| `hourly_rate (USD)` | -37.5 – 142.5 | 0 |
| `rating` | -2.2 – 7.4 | 0 |


### 5.3 — Outlier Detection Summary

| Column | Method | Outliers found | Decision | Reason |
|---|---|---|---|---|
| `age` | IQR | 0 | — | No values outside plausible range |
| `years_of_experience` | IQR | 9 (values 39-41) | **Retain** | Physically plausible, not errors; capping/removing would delete legitimate veteran freelancers |
| `hourly_rate (USD)` | IQR | 0 | — | No values outside plausible range |
| `rating` | IQR | 0 | — | Bounded 0-5 by design, can't produce outliers |


Only `years_of_experience` flags anything. The other three columns' real-world values simply don't stretch far enough to cross their IQR bounds, reported honestly rather than forcing a finding.

---

## Step: 6 Data Type Correction

This step is a formal verification, not new work — the dtype fixes for this dataset already happened as side effects of earlier steps (the `hourly_rate` parsing in Step 2, the `int` casts on `age`/`years_of_experience` also in Step 2). What hasn't happened yet is *checking every column explicitly* against what it should be, rather than assuming it's fine because nothing broke.

In [22]:
expected_dtypes={
    'freelancer_ID':'object (string identifier and not numeric)',
    'name': 'object (string)',
    'gender': 'object (string, standardized)',
    'age': 'int64',
    'country': 'ibject (string)',
    'language': 'object (string)',
    'primary_skill': 'object (string)',
    'years_of_experience': 'int64',
    'hourly_rate (USD)': 'float64',
    'rating': 'float64',
    'has_rating': 'bool',
}

check_dtype=pd.DataFrame({
    'actual_dtype': df.dtypes.astype(str),
    'expected_dtype': pd.Series(expected_dtypes)

})
check_dtype

,actual_dtype,expected_dtype
freelancer_ID,str,object (string identifier and not numeric)
name,str,object (string)
gender,str,"object (string, standardized)"
age,int64,int64
country,str,ibject (string)
language,str,object (string)
primary_skill,str,object (string)
years_of_experience,int64,int64
hourly_rate (USD),float64,float64
rating,float64,float64


### Verification Result
- All **11** columns matches their expected dtypes.
- `freelancer_ID` looks numeric-adjacent (`FL250001`) but is deliberately kept as a string. IDs are labels, not quantities .

## Step:7 Before Vs After Summary:
Comparing the dataset's state at load time (Step 1) against its state now, after all cleaning steps.

In [23]:
raw_dataset= pd.read_csv("global_freelancers_raw.csv")
raw_dataset=raw_dataset.drop(columns=["is_active", "client_satisfaction"])

before_state={
    'row_count': len(raw_dataset),
    'null_count_total': int(raw_dataset.isna().sum().sum()),
    'duplicate_rows': int(raw_dataset.duplicated().sum()),
    'dtype_issues': 1, # hourly_rate (USD) only dtype stored as string. step 1
}

after_state={
    'row_count': len(df),
    'null_count_total': int(df.isna().sum().sum()),
    'duplicate_rows': int(df.duplicated().sum()),
    'dtype_issues': 0, # verified in step 6, all columns matched expected dtypes
}

summary_table= pd.DataFrame({"Before": before_state, "After": after_state})
summary_table

,Before,After
row_count,1000,1000
null_count_total,276,101
duplicate_rows,0,0
dtype_issues,1,0


## Step:8 Save Cleaned Dataset

In [24]:
output_path = 'global_freelancers_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Final shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Saved: global_freelancers_cleaned.csv
Final shape: 1000 rows x 11 columns


,freelancer_ID,name,gender,age,country,language,primary_skill,years_of_experience,hourly_rate (USD),rating,has_rating
0,FL250001,Ms. Nicole Kidd,Female,52,Italy,Italian,Blockchain Development,11,100.0,NaN,False
1,FL250002,Vanessa Garcia,Female,52,Australia,English,Mobile Apps,34,100.0,3.3,True
2,FL250003,Juan Nelson,Male,53,Germany,German,Graphic Design,31,50.0,0.0,True
3,FL250004,Amanda Spencer,Female,38,Australia,English,Web Development,4,40.0,1.5,True
4,FL250005,Lynn Curtis DDS,Female,53,Germany,German,Web Development,27,30.0,4.8,True


In [27]:
check = pd.read_csv("global_freelancers_cleaned.csv")
print(check['hourly_rate (USD)'].isna().sum())   # should be 0
print(check['rating'].isna().sum())              # should be 101
print(check.shape)                                # should be (1000, 11)

0
101
(1000, 11)
